# XMCA-Net-Lite: 04 - Perception Consistency Scoring (S_P)
### Purpose
Compute visual Perception Consistency Score (S_P):
1. Extract atomic claims from generated answers.
2. Detect visual regions via OWL-ViT.
3. Compute region-to-claim similarity using SigLIP/CLIP.
Outputs saved in Parquet shards.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
# Robust project root discovery (supports Kaggle uppercase/lowercase and local)
PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/XMCA"), Path("/kaggle/working/xmca")]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.perception.claims import extract_claims_from_answer
from src.perception.detector import OwlViTDetector
from src.perception.scorer import PerceptionScorer
from src.utils.checkpoints import ShardManager
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Load all Qwen2-VL generation shards
qwen_gen_dir = Path(config["paths"]["generations_dir"]) / "qwen2vl"
shard_mgr_qwen = ShardManager(qwen_gen_dir)
gen_df = shard_mgr_qwen.load_all_shards("qwen2vl_generation")
print(f"Loaded {len(gen_df)} generation items for perception evaluation.", flush=True)

detector = OwlViTDetector(
    model_id=config["models"]["perception"]["detector_model_id"],
    box_threshold=config["models"]["perception"]["box_threshold"]
)
scorer = PerceptionScorer(model_id=config["models"]["perception"]["similarity_model_id"])

out_dir = Path(config["paths"]["perception_dir"])
shard_mgr_sp = ShardManager(out_dir, shard_size=config["execution"]["shard_size"])
items = gen_df.to_dict(orient="records")
remaining = shard_mgr_sp.get_remaining_items(items, id_key="id")
total_rem = len(remaining)
print(f"Remaining items to compute S_P: {total_rem}", flush=True)

shard_records = []
shard_idx = 0

for i, row in enumerate(remaining, 1):
    claims = extract_claims_from_answer(row["generated_answer"])
    boxes, labels, confs = detector.detect_regions(row.get("image_path", "dummy.jpg"), claims)
    s_p = scorer.compute_s_p(row.get("image_path", "dummy.jpg"), claims, boxes)
    
    shard_records.append({
        "id": row["id"],
        "dataset": row["dataset"],
        "claims": claims,
        "boxes": boxes,
        "labels": labels,
        "s_p": s_p,
        "is_hallucination_gt": row["is_hallucination_gt"]
    })
    
    if i % 50 == 0 or i == total_rem:
        pct = (i / total_rem) * 100
        print(f"Perception scoring (S_P): {i}/{total_rem} done ({pct:.1f}%) [Shard {shard_idx}]", flush=True)

    if len(shard_records) >= config["execution"]["shard_size"]:
        shard_mgr_sp.save_shard(shard_idx, shard_records, prefix="perception_scores")
        print(f"✓ Saved perception shard {shard_idx:04d} ({len(shard_records)} items saved, {i}/{total_rem} done).", flush=True)
        shard_records = []
        shard_idx += 1

if shard_records:
    shard_mgr_sp.save_shard(shard_idx, shard_records, prefix="perception_scores")
    print(f"✓ Saved final perception shard {shard_idx:04d} ({total_rem}/{total_rem} done).", flush=True)

save_run_metadata(out_dir, config, seed, "04_perception", {"total_processed": len(items)})
print("
Step 04 completed. Perception scores saved.", flush=True)